# Partie G - Comparaison finale, decision, questions et livrables



## 0. Environnement et regles du TP

Le decoupage officiel train/test est conserve. Le test officiel ne sert qu'a l'evaluation finale. Toutes les decisions sont prises sur la validation.

In [2]:
# Configuration simple
RANDOM_STATE = 42

import random
import warnings

import numpy as np
import pandas as pd

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
warnings.filterwarnings('ignore')

try:
    import torch
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Peripherique :', DEVICE)
    if DEVICE.type == 'cuda':
        print('GPU :', torch.cuda.get_device_name(0))
except ImportError:
    torch = None
    DEVICE = 'cpu'
    print('PyTorch indisponible : calcul CPU.')

from sklearn.metrics import (
    f1_score, average_precision_score, confusion_matrix
)

RESULTS = []

def add_result(name, y_true, scores, threshold=0.5, split='validation'):
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores)
    pred = (scores >= threshold).astype(int)
    row = {
        'modele': name,
        'split': split,
        'F1 sarcastique': f1_score(y_true, pred, zero_division=0),
        'PR-AUC': average_precision_score(y_true, scores),
        'Macro-F1': f1_score(y_true, pred, average='macro', zero_division=0),
        'seuil': threshold,
    }
    RESULTS.append(row)
    return row


Peripherique : cuda
GPU : NVIDIA GeForce RTX 2050


In [3]:
BASE = 'https://raw.githubusercontent.com/iabufarha/ArSarcasm-v2/main/ArSarcasm-v2/'
full = pd.read_csv(BASE + 'training_data.csv')
test = pd.read_csv(BASE + 'testing_data.csv')
full['id'] = range(len(full))
test['id'] = range(len(test))
print('Train officiel :', full.shape)
print('Test officiel  :', test.shape)
display(full.head())


Train officiel : (12548, 5)
Test officiel  : (3000, 5)


,tweet,sarcasm,sentiment,dialect,id
0,"""د. #محمود_العلايلي:أرى أن الفريق #أحمد_شفيق ر...",False,NEU,msa,0
1,"""مع فيدرر يا آجا والكبار 😍 https://t.co/hrBeHb...",False,NEU,msa,1
2,“الداعون لمبدأ الاختلاط بين الجنسين؛ كالداعين ...,True,NEG,msa,2
3,"""@ihe_94 @ya78m @amooo5 @badiajnikhar @Oukasaf...",True,NEG,gulf,3
4,"""قل شرق حلب ولا تقل حلب الشرقية ....وقل غرب حل...",False,NEU,msa,4


# Partie G - Comparaison finale et decision

## G1. Tableau final

Inclure F1, PR-AUC, Macro-F1, robustesse, temps d'entrainement, latence CPU par tweet, taille du modele, parametres entraines et generalisation.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

full['text_clean'] = full['tweet'].fillna('').astype(str)
tr, va = train_test_split(full, test_size=0.2, stratify=full['sarcasm'], random_state=RANDOM_STATE)
word_model = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=30000)),
    ('clf', LogisticRegression(max_iter=300, class_weight='balanced'))
])
word_model.fit(tr['text_clean'], tr['sarcasm'].astype(int))
y_val = va['sarcasm'].astype(int).to_numpy()
word_scores = word_model.predict_proba(va['text_clean'])[:, 1]
thresholds = np.arange(0.1, 0.91, 0.05)
best_threshold = max(thresholds, key=lambda t: f1_score(y_val, word_scores >= t, zero_division=0))


In [5]:
results_table = pd.DataFrame(RESULTS)
if not results_table.empty:
    display(results_table.sort_values(['split', 'F1 sarcastique'], ascending=[True, False]))
    print('Aucun resultat disponible.')


## G2. Matrice de confusion et decision de production

In [6]:
cm = confusion_matrix(y_val, word_scores >= best_threshold)
display(pd.DataFrame(cm, index=['vrai non sarcastique', 'vrai sarcastique'], columns=['predit non sarcastique', 'predit sarcastique']))
print('Decision : choisir le modele selon F1, PR-AUC, robustesse, cout et latence.')


,predit non sarcastique,predit sarcastique
vrai non sarcastique,1678,398
vrai sarcastique,179,255


Decision : choisir le modele selon F1, PR-AUC, robustesse, cout et latence.
